In [26]:
import geopandas as gpd
import pandas as pd
from pathlib import Path

In [23]:
input_dir = Path.cwd() / "input"
output_dir = Path.cwd() / "output"

In [85]:
primair = input_dir / "Primaire_keringen" / "geo_infrastructuur_primaire_keringen.shp"
niet_primair = (
    input_dir
    / "Niet_primaire_keringen"
    / "geo_infrastructuur_niet_primaire_keringen.shp"
)
doorbraak_locaties_primair = (
    input_dir
    / "Doorbraaklocaties"
    / "geo_gebiedsindeling_doorbraaklocaties_primair.shp"
)
doorbraak_locaties_regionaal = (
    input_dir
    / "Doorbraaklocaties"
    / "geo_gebiedsindeling_doorbraaklocaties_regionaal.shp"
)

dijkringen = input_dir / "Dijkringen" / "geo_infrastructuur_dijkringen.shp"
geometrie_35_3 = input_dir / "Geometrie_voor_Traject_#35-3" / "Normtrajecten_trajectdelen_v26092022.shp"

In [188]:
scenario_list = pd.read_excel(input_dir / "scenariolist.xlsx")
scenario_manager = pd.read_excel(input_dir / "scenmanager_compare.xlsx", sheet_name="kansen_trajecten_aangepast")

In [189]:
gdf_primair = gpd.read_file(primair)
gdf_niet_primair = gpd.read_file(niet_primair)
gdf_doorbraak_locaties_primair = gpd.read_file(doorbraak_locaties_primair)
gdf_doorbraak_locaties_regionaal = gpd.read_file(doorbraak_locaties_regionaal)
gdf_geometrie_35_3 = gpd.read_file(geometrie_35_3)

gdf_dijkringen = gpd.read_file(dijkringen)
gdf_primair.set_index(['TRAJECT_ID','TRAJECT_DL'], inplace=True)

Klein zijspoor voor traject 35-3, hier is de geometries verloren gegaan. In de oude versie ging dit om 36-0 maar dat is dezelfde als 36-1

In [190]:
set(scenario_manager.set_index(['TRAJECT_ID','TRAJECT_DL']).index).difference(set(gdf_primair.index))

{('#35-3', '036013')}

In [191]:
print(scenario_manager.set_index(['TRAJECT_ID','TRAJECT_DL']).loc[('#35-3', '036013'),'WIJZ_UPDATE_2022'])

Cascade, dus geen officieel traject(deel). Norm gelijk aan 1/3000 per jaar, zijnde de meest soepele trajectnorm vanuit de dijkring 36 (dijkring 35 kan vanuit meerdere breslocaties in dijkring 36 overstromen; niet alleen vanuit keent/kraaijenbergse plas).


In [192]:
scenario_manager.set_index(['TRAJECT_ID','TRAJECT_DL']).loc[('#35-3', '036013')][['LENGTE_DL', 'KANSDL_2022']]

LENGTE_DL      355.488875
KANSDL_2022         562.0
Name: (#35-3, 036013), dtype: object

In [193]:
gdf_36_0 = gdf_geometrie_35_3[gdf_geometrie_35_3['TRAJECT_ID'] == '36-0'].set_index(['TRAJECT_ID','TRAJECT_DL'])

gdf_36_0_adj = gdf_36_0[['LENGTE_TRA','DUIDING205','KLASSE2050','KANS2019','geometry']].copy()
gdf_36_0_adj.rename(columns={'LENGTE_TRA': 'Shape_Leng', 'DUIDING205': 'DUIDING', 'KLASSE2050': 'klasse', 'KANS2019': 'KANS'}, inplace=True)

# neem de lans over uit de scenario manager
gdf_36_0_adj.loc[('36-0', '036013'),'KANS'] = scenario_manager.set_index(['TRAJECT_ID','TRAJECT_DL']).loc[('#35-3', '036013')]['KANSDL_2022']

gdf_36_0_adj.index = pd.MultiIndex.from_tuples([('#35-3', '036013')], names=['TRAJECT_ID','TRAJECT_DL'])
gdf_36_0_adj

,,Shape_Leng,DUIDING,klasse,KANS,geometry
TRAJECT_ID,TRAJECT_DL,,,,,
#35-3,036013,355.488875,Zeer kleine kans: 1/3.000 tot 1/30.000 per jaar,4.0,562.0,"LINESTRING (134268.035 412893.105, 134516.744 ..."


In [194]:
gdf_primair_comb = gpd.GeoDataFrame(pd.concat([gdf_primair, gdf_36_0_adj]))

In [195]:
df_kansen_2025 = scenario_manager.set_index(['TRAJECT_ID','TRAJECT_DL'])['NORMOG_2050']
gdf_primair_kansen = gdf_primair_comb.join(df_kansen_2025, how='inner')

In [182]:
gdf_primair_kansen

Shape_Leng  \
TRAJECT_ID TRAJECT_DL                
1-1        001002      9160.156374   
1-2        001001      3915.212676   
10-1       010014      5651.075246   
           010013      1724.460675   
           010016      2426.830269   
...                            ...   
#49-3      049006b      144.497675   
13b-1      013b01      3311.632966   
11-2       011001      6332.489981   
#44-3      044012        96.423486   
#35-3      036013       355.488875   

                                                               DUIDING  \
TRAJECT_ID TRAJECT_DL                                                    
1-1        001002             Extreem kleine kans: < 1/30.000 per jaar   
1-2        001001              Kleine kans: 1/300 tot 1/3.000 per jaar   
10-1       010014      Zeer kleine kans: 1/3.000 tot 1/30.000 per jaar   
           010013              Kleine kans: 1/300 tot 1/3.000 per jaar   
           010016      Zeer kleine kans: 1/3.000 tot 1/30.000 per jaar   
...                                                                ...   
#49-3      049006b             Kleine kans: 1/300 tot 1/3.000 per jaar   
13b-1      013b01      Zeer kleine kans: 1/3.000 tot 1/30.000 per jaar   
11-2       011001             Extreem kleine kans: < 1/30.000 per jaar   
#44-3      044012             Extreem kleine kans: < 1/30.000 per jaar   
#35-3      036013      Zeer kleine kans: 1/3.000 tot 1/30.000 per jaar   

                       klasse       KANS  \
TRAJECT_ID TRAJECT_DL                      
1-1        001002         5.0  7874016.0   
1-2        001001         3.0      358.0   
10-1       010014         4.0     3115.0   
           010013         3.0     1042.0   
           010016         4.0    11099.0   
...                       ...        ...   
#49-3      049006b        3.0      917.0   
13b-1      013b01         4.0    10000.0   
11-2       011001         5.0    70423.0   
#44-3      044012         5.0   400000.0   
#35-3      036013         4.0      562.0   

                                                                geometry  \
TRAJECT_ID TRAJECT_DL                                                      
1-1        001002      LINESTRING (209653.000 610745.000, 209653.509 ...   
1-2        001001      LINESTRING (206219.000 609839.000, 206219.236 ...   
10-1       010014      LINESTRING (202231.127 511585.388, 202211.844 ...   
           010013      LINESTRING (200717.949 515831.240, 200702.531 ...   
           010016      LINESTRING (200190.454 502566.659, 200205.078 ...   
...                                                                  ...   
#49-3      049006b     LINESTRING (212068.990 457232.110, 212213.453 ...   
13b-1      013b01      LINESTRING (135599.938 497172.500, 135599.766 ...   
11-2       011001      LINESTRING (187009.777 510526.344, 187014.157 ...   
#44-3      044012      LINESTRING (116790.830 494711.674, 116880.127 ...   
#35-3      036013      LINESTRING (134268.035 412893.105, 134516.744 ...   

                       KANS_NORM_2025  
TRAJECT_ID TRAJECT_DL                  
1-1        001002                1000  
1-2        001001                1000  
10-1       010014                1000  
           010013                1000  
           010016                1000  
...                               ...  
#49-3      049006b               3000  
13b-1      013b01                 100  
11-2       011001                1000  
#44-3      044012              100000  
#35-3      036013                3000  

[721 rows x 6 columns]

In [183]:
layers = [
    gdf_primair_kansen,
    gdf_niet_primair,
    gdf_doorbraak_locaties_primair,
    gdf_doorbraak_locaties_regionaal,
    gdf_dijkringen,
]
for layer in layers:
    for col in ['geom','notify','Shape_Leng','LT30','F30T300','F300T3000','F3000T30K','GT30K','sign','KENMERK_V2','keuze','cascade']:
        if col in layer.columns:
            layer.drop(columns=[col], inplace=True)
    

In [ ]:
# standaardiseren van kolomnamen
gdf_primair_kansen.rename()

In [ ]:
display(gdf_primair_kansen.head())
display(gdf_niet_primair.head())
display(gdf_doorbraak_locaties_primair.head())
display(gdf_doorbraak_locaties_regionaal.head())
display(gdf_dijkringen.head())

# display(gdf_primair.head())


DUIDING  \
TRAJECT_ID TRAJECT_DL                                                    
1-1        001002             Extreem kleine kans: < 1/30.000 per jaar   
1-2        001001              Kleine kans: 1/300 tot 1/3.000 per jaar   
10-1       010014      Zeer kleine kans: 1/3.000 tot 1/30.000 per jaar   
           010013              Kleine kans: 1/300 tot 1/3.000 per jaar   
           010016      Zeer kleine kans: 1/3.000 tot 1/30.000 per jaar   

                       klasse       KANS  \
TRAJECT_ID TRAJECT_DL                      
1-1        001002         5.0  7874016.0   
1-2        001001         3.0      358.0   
10-1       010014         4.0     3115.0   
           010013         3.0     1042.0   
           010016         4.0    11099.0   

                                                                geometry  \
TRAJECT_ID TRAJECT_DL                                                      
1-1        001002      LINESTRING (209653.000 610745.000, 209653.509 ...   
1-2        001001      LINESTRING (206219.000 609839.000, 206219.236 ...   
10-1       010014      LINESTRING (202231.127 511585.388, 202211.844 ...   
           010013      LINESTRING (200717.949 515831.240, 200702.531 ...   
           010016      LINESTRING (200190.454 502566.659, 200205.078 ...   

                       KANS_NORM_2025  
TRAJECT_ID TRAJECT_DL                  
1-1        001002                1000  
1-2        001001                1000  
10-1       010014                1000  
           010013                1000  
           010016                1000

,NAAM,Beheerder,kansk,Provincie,Type,Soort,Normfreq,Opmerkinge,geometry
0,Meppelerdiep,Drents Overijsselse Delta,2,Overijssel,regionaal - ander water kerend,dijk/kade,1:100,None,"LINESTRING Z (207512.884 522906.352 0.000, 207..."
1,Meppelerdiep,Drents Overijsselse Delta,2,Overijssel,regionaal - ander water kerend,dijk/kade,1:100,None,"LINESTRING Z (206710.507 521965.527 0.000, 206..."
2,Meppelerdiep,Drents Overijsselse Delta,2,Overijssel,regionaal - ander water kerend,dijk/kade,1:100,None,"LINESTRING Z (211454.623 522058.751 0.000, 211..."
3,Meppelerdiep,Drents Overijsselse Delta,2,Overijssel,regionaal - ander water kerend,dijk/kade,1:100,None,"LINESTRING Z (201338.768 517152.194 0.000, 201..."
4,Sallandse weteringen,Drents Overijsselse Delta,2,Overijssel,regionaal - ander water kerend,dijk/kade,1:100,None,"LINESTRING Z (202451.098 503283.356 0.000, 202..."


,gid,id,dijkringnr,naam,code,x,y,geometry
0,1,1,1,Jachthaven,PRIM,206791.0,609827.0,POINT (206791.000 609827.000)
1,2,2,1,Noordzee,PRIM,205564.0,611746.0,POINT (205564.000 611746.000)
2,3,3,1,Maximaal Scenario,PRIM,206960.0,610870.0,POINT (206960.000 610870.000)
3,4,4,2,Nes,PRIM,180600.0,606000.0,POINT (180600.000 606000.000)
4,5,5,2,Hollum,PRIM,172274.0,604623.0,POINT (172274.000 604623.000)


,gid,id,regio,naam,code,x,y,geometry
0,699,699,Waterlandseboezem,Noordhollandsch Kanaal 3249,REG,125218.0186,499690.6223,POINT (125218.019 499690.622)
1,700,700,Waterlandseboezem,Noordhollandsch Kanaal 3736,REG,125142.9871,499215.6273,POINT (125142.987 499215.627)
2,701,701,Waterlandseboezem,Noordhollandsch Kanaal 4162,REG,125121.4076,498788.1023,POINT (125121.408 498788.102)
3,702,702,Waterlandseboezem,Noordhollandsch Kanaal 4587,REG,125109.9980,498362.8220,POINT (125109.998 498362.822)
4,703,703,Waterlandseboezem,Noordhollandsch Kanaal 5018,REG,125118.5537,497934.5432,POINT (125118.554 497934.543)


,OBJECTID,DIJKRINGNR,DIJKRING,BEHEERDER,geometry
0,85,77,Roermond 3,Waterschap Roer en Overmaas,"POLYGON ((194334.266 355701.656, 194364.406 35..."
1,82,75,Buggenum,Waterschap Peel en Maasvallei,"POLYGON ((196463.344 360759.781, 196569.516 36..."
2,86,76a,Roermond 2,Waterschap Roer en Overmaas,"POLYGON ((196070.453 355602.000, 196179.547 35..."
3,84,76c,Roermond 4,Waterschap Roer en Overmaas,"POLYGON ((196671.719 356818.406, 196701.594 35..."
4,81,74,Neer,Waterschap Peel en Maasvallei,"POLYGON ((197906.359 363683.406, 198061.922 36..."


In [186]:
driver = "GPKG"
gdf_primair_kansen.to_file(
    output_dir / "kansendatabase.gpkg", driver=driver, layer="primaire_keringen"
)
gdf_niet_primair.to_file(
    output_dir / "kansendatabase.gpkg", driver=driver, layer="niet_primaire_keringen"
)
gdf_doorbraak_locaties_primair.to_file(
    output_dir / "kansendatabase.gpkg",
    driver=driver,
    layer="doorbraak_locaties_primair",
)
gdf_doorbraak_locaties_regionaal.to_file(
    output_dir / "kansendatabase.gpkg",
    driver=driver,
    layer="doorbraak_locaties_regionaal",
)
gdf_dijkringen.to_file(
    output_dir / "kansendatabase.gpkg", driver=driver, layer="dijkringen"
)